# Build a printable weekly schedule from DonSheet

This notebook reads the DonSheet workbook, normalizes scheduled meetings, groups cross-listed classes that share the same room/time slot, and exports a letter-sized printable schedule workbook.

In [ ]:
import re
import pandas as pd
import openpyxl
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Border, Side, Alignment
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.page import PageMargins

In [ ]:
INPUT_FILE = "DonSheet_202641_Fall_v1.0.xlsx"
OUTPUT_FILE = "DonSheet_printable_week_schedule.xlsx"

In [ ]:
def is_valid_time(val):
    return bool(re.fullmatch(r"\d{4}", str(val).strip()))

def fmt_time(s):
    s = str(s).zfill(4)
    hh, mm = int(s[:2]), int(s[2:])
    ampm = "AM" if hh < 12 else "PM"
    hh12 = hh % 12 or 12
    return f"{hh12}:{mm:02d} {ampm}"

def fmt_range(start, end):
    return f"{fmt_time(start)}–{fmt_time(end)}"

def uniq(seq):
    out, seen = [], set()
    for x in seq:
        if pd.isna(x):
            continue
        s = str(x).strip()
        if not s or s in seen:
            continue
        seen.add(s)
        out.append(s)
    return out

def pick_title(row):
    sec = str(row.get("Section Title") or "").strip()
    cat = str(row.get("Catalog Title") or "").strip()
    return sec if sec and sec.lower() not in {"none", cat.lower()} else cat

def natural_room_key(room):
    m = re.match(r"([A-Z]+)(\d+)", str(room))
    return (m.group(1), int(m.group(2))) if m else (str(room), 0)

In [ ]:
# 1) Read every department sheet into one dataframe
wb = openpyxl.load_workbook(INPUT_FILE, data_only=True)
ignore = {"DropDownMenu", "InstructorMap"}

records = []
for sheet_name in wb.sheetnames:
    if sheet_name in ignore:
        continue
    ws = wb[sheet_name]
    headers = [ws.cell(1, c).value for c in range(1, ws.max_column + 1)]
    for r in range(2, ws.max_row + 1):
        row = {headers[c - 1]: ws.cell(r, c).value for c in range(1, ws.max_column + 1)}
        if row.get("CRN") is None and row.get("Catalog Title") is None:
            continue
        row["_sheet"] = sheet_name
        records.append(row)

df = pd.DataFrame(records)
print(df.shape)
df.head()

In [ ]:
# 2) Keep only room-based classes with valid times, then explode to one row per meeting-day
dedupe_cols = [
    "CRN", "Subject", "Course Number {Crse Num}", "Course Section {Seq Crse Num}",
    "Catalog Title", "Section Title", "Course Beginning Time {Meet Beg Time}",
    "Course Ending Time {Meet End Time}", "MON", "TUE", "WED", "THU", "FRI", "SAT", "SUN",
    "Room {Meet Room}", "Instructor Name {Instr Name}"
]

valid = df[
    df["Course Beginning Time {Meet Beg Time}"].map(is_valid_time)
    & df["Course Ending Time {Meet End Time}"].map(is_valid_time)
    & df["Room {Meet Room}"].notna()
].copy().drop_duplicates(subset=dedupe_cols)

day_map = [
    ("MON", "Monday", "M"),
    ("TUE", "Tuesday", "T"),
    ("WED", "Wednesday", "W"),
    ("THU", "Thursday", "R"),
    ("FRI", "Friday", "F"),
    ("SAT", "Saturday", "S"),
    ("SUN", "Sunday", "U"),
]

meeting_rows = []
for _, row in valid.iterrows():
    for day_col, day_name, marker in day_map:
        if str(row.get(day_col) or "").strip() == marker:
            meeting_rows.append({
                "Day": day_name,
                "Room": str(row["Room {Meet Room}"]).strip(),
                "Start": str(row["Course Beginning Time {Meet Beg Time}"]).zfill(4),
                "End": str(row["Course Ending Time {Meet End Time}"]).zfill(4),
                "Course Code": f"{row['Subject']} {row['Course Number {Crse Num}']}-{row['Course Section {Seq Crse Num}']}",
                "Title": pick_title(row),
                "Catalog Title": str(row.get("Catalog Title") or "").strip(),
                "CRN": str(row.get("CRN") or "").strip(),
                "Instructor": str(row.get("Instructor Name {Instr Name}") or "").strip(),
                "Credits": str(row.get("Credits {Sect Crs}") or "").strip(),
                "Enrollment Cap": str(row.get("Enrollment Cap {Max Enrl}") or "").strip(),
            })

meetings = pd.DataFrame(meeting_rows)
print(meetings.shape)
meetings.head()

In [ ]:
# 3) Group cross-listed / stacked classes that share the same room/day/time
grouped_rows = []
for keys, g in meetings.groupby(["Day", "Room", "Start", "End"], sort=False):
    codes = uniq(g["Course Code"])
    titles = uniq(g["Title"])
    catalog_titles = uniq(g["Catalog Title"])
    if len(titles) > 2:
        titles = catalog_titles[:2]
    instrs = uniq(g["Instructor"])
    crns = [c for c in uniq(g["CRN"]) if c not in {"###", "####"}]
    grouped_rows.append({
        "Day": keys[0],
        "Room": keys[1],
        "Start": keys[2],
        "End": keys[3],
        "Course Codes": " / ".join(codes),
        "Title": " / ".join(titles),
        "Instructor": " / ".join(instrs),
        "CRNs": ", ".join(crns),
        "Time": fmt_range(keys[2], keys[3]),
    })

slots = pd.DataFrame(grouped_rows).sort_values(["Day", "Room", "Start", "Course Codes"]).reset_index(drop=True)
slots.head(10)

In [ ]:
# 4) Build the printable workbook
rooms = sorted(slots["Room"].dropna().unique(), key=natural_room_key)
days = [d for d in ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"] if d in set(slots["Day"])]

out = Workbook()
ws = out.active
ws.title = "Weekly Schedule"
ws.sheet_view.showGridLines = False

dark = "1F4E78"
header_fill = PatternFill("solid", fgColor=dark)
day_fill = PatternFill("solid", fgColor="DDEBF7")
sub_fill = PatternFill("solid", fgColor="F2F6FB")
thin_gray = Side(style="thin", color="B7C9D6")
med_dark = Side(style="medium", color=dark)
border = Border(left=thin_gray, right=thin_gray, top=thin_gray, bottom=thin_gray)

subject_fills = {
    "CHIS": "FFF2CC", "DSLE": "E2F0D9", "GSEM": "D9EAF7", "MSSN": "FCE4D6",
    "NTST": "EADCF8", "OTST": "DDEBF7", "PATH": "E4DFEC", "THST": "F4CCCC", "ANEA": "FFF2F2",
}

def slot_subject(codes):
    return codes.split()[0] if codes else ""

for i, room in enumerate(rooms, start=1):
    ws.column_dimensions[get_column_letter(i)].width = 16.5

last_col = len(rooms)
last_col_letter = get_column_letter(last_col)

ws.merge_cells(start_row=1, start_column=1, end_row=1, end_column=last_col)
ws["A1"] = "Printable Weekly Schedule"
ws["A1"].fill = header_fill
ws["A1"].font = Font(color="FFFFFF", bold=True, size=15)
ws["A1"].alignment = Alignment(horizontal="center", vertical="center")
ws.row_dimensions[1].height = 24

ws.merge_cells(start_row=2, start_column=1, end_row=2, end_column=last_col)
ws["A2"] = "Generated from DonSheet • grouped by room/day/time • fit to one landscape letter page"
ws["A2"].fill = sub_fill
ws["A2"].font = Font(color="666666", size=8, italic=True)
ws["A2"].alignment = Alignment(horizontal="center", vertical="center")
ws.row_dimensions[2].height = 18

for c, room in enumerate(rooms, start=1):
    cell = ws.cell(row=3, column=c, value=room)
    cell.fill = header_fill
    cell.font = Font(color="FFFFFF", bold=True, size=10)
    cell.alignment = Alignment(horizontal="center", vertical="center")
    cell.border = Border(top=med_dark, bottom=med_dark, left=thin_gray, right=thin_gray)

current_row = 4
for day in days:
    day_slots = slots[slots["Day"] == day]
    if day_slots.empty:
        continue

    ws.merge_cells(start_row=current_row, start_column=1, end_row=current_row, end_column=last_col)
    cell = ws.cell(row=current_row, column=1, value=day)
    cell.fill = day_fill
    cell.font = Font(bold=True, size=11)
    cell.alignment = Alignment(horizontal="left", vertical="center")
    cell.border = Border(top=med_dark, bottom=thin_gray)
    ws.row_dimensions[current_row].height = 18
    current_row += 1

    room_lists = {
        room: day_slots[day_slots["Room"] == room].sort_values(["Start", "Course Codes"]).to_dict("records")
        for room in rooms
    }
    max_len = max(len(v) for v in room_lists.values())

    for i in range(max_len):
        for cidx, room in enumerate(rooms, start=1):
            cell = ws.cell(row=current_row, column=cidx)
            cell.border = border
            cell.alignment = Alignment(horizontal="left", vertical="top", wrap_text=True)
            recs = room_lists[room]
            if i < len(recs):
                rec = recs[i]
                cell.value = "\n".join([
                    rec["Course Codes"] + (f"  (CRN {rec['CRNs']})" if rec["CRNs"] else ""),
                    rec["Title"],
                    f"{rec['Time']} • {rec['Instructor']}",
                ])
                cell.fill = PatternFill("solid", fgColor=subject_fills.get(slot_subject(rec["Course Codes"]), "FFFFFF"))
                cell.font = Font(size=8)
            else:
                cell.value = ""
            ws.row_dimensions[current_row].height = 42
        current_row += 1

    ws.row_dimensions[current_row].height = 8
    current_row += 1

ws.page_setup.orientation = ws.ORIENTATION_LANDSCAPE
ws.page_setup.paperSize = ws.PAPERSIZE_LETTER
ws.page_setup.fitToWidth = 1
ws.page_setup.fitToHeight = 1
ws.sheet_properties.pageSetUpPr.fitToPage = True
ws.page_margins = PageMargins(left=0.25, right=0.25, top=0.35, bottom=0.35, header=0.15, footer=0.15)
ws.print_area = f"A1:{last_col_letter}{current_row}"

out.save(OUTPUT_FILE)
print(f"Saved {OUTPUT_FILE}")